In [1]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244081127

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] exp_env
[学号]       20244081127
[机器指纹] a8365f4adeced7e9
[RUN_ID]   20244081127-a8365f4adeced7e9


## Python 正则表达式练习

In [2]:
import re

# 任务1 检查字符串是否仅包含英文字母a-z、A-Z和数字0-9
def check_alnum(s):
    pattern = r'^[a-zA-Z0-9]+$'
    return bool(re.fullmatch(pattern, s))

print(check_alnum("abc123"))  
print(check_alnum("abc_123")) 
print(check_alnum(""))        

True
False
False


In [3]:
# 任务2 匹配一个a后面跟着零个或多个b的字符串
def match_a_zero_b(s):
    pattern = r'^ab*$'
    return bool(re.fullmatch(pattern, s))

print(match_a_zero_b("a"))    
print(match_a_zero_b("abb"))  
print(match_a_zero_b("acb"))  

True
True
False


In [4]:
# 任务3 匹配一个a后面跟着一个或多个b的字符串
def match_a_oneplus_b(s):
    pattern = r'^ab+$'
    return bool(re.fullmatch(pattern, s))

print(match_a_oneplus_b("ab")) 
print(match_a_oneplus_b("a")) 
print(match_a_oneplus_b("abb")) 

True
False
True


In [5]:
# 任务4 匹配以a开头、以b结尾，且中间可以有任意字符的字符串
def match_start_a_end_b(s):
    pattern = r'^a.*b$'
    return bool(re.fullmatch(pattern, s))

print(match_start_a_end_b("a123b")) 
print(match_start_a_end_b("ab"))  
print(match_start_a_end_b("axc")) 

True
True
False


In [6]:
# 任务5 删除IP地址中各段数字的前导零；某段全为零时保留一个0
def ip_remove_leading_zero(ip_str):
    def sub_zero(seg):
        return re.sub(r'^0+(?=\d)', '', seg)
    return '.'.join([sub_zero(part) for part in ip_str.split('.')])

print(ip_remove_leading_zero("192.168.001.010")) 
print(ip_remove_leading_zero("000.010.00.1"))

192.168.1.10
0.10.0.1


In [7]:
# 任务6 将字符串中的空格替换为下划线，并将下划线替换为空格
def swap_space_underline(s):
    # 先把空格临时替换
    s1 = re.sub(r' ', '\x01', s)
    s2 = re.sub(r'_', ' ', s1)
    s3 = re.sub(r'\x01', '_', s2)
    return s3

print(swap_space_underline("hello world_test")) 

hello_world test


In [8]:
# 任务7 将 yyyy-mm-dd 格式的日期转换为 dd-mm-yyyy
def date_ymd2dmy(date_str):
    pattern = r'^(\d{4})-(\d{2})-(\d{2})$'
    return re.sub(pattern, r'\3-\2-\1', date_str)

print(date_ymd2dmy("2026-11-22")) 

22-11-2026


In [9]:
# 任务8 找出给定字符串中所有以a或e开头的单词
def find_word_start_ae(s):
    return re.findall(r'\b[ae]\w*\b', s)

print(find_word_start_ae("apple egg cat elephant bear"))

['apple', 'egg', 'elephant']


In [10]:
# 任务9 将字符串中的 Road 缩写为 Rd
def road2rd(s):
    return re.sub(r'\bRoad\b', 'Rd', s)

print(road2rd("Main Road, Roadside")) 

Main Rd, Roadside


In [11]:
# 任务10 找出字符串中所有长度至少为4个字符的单词
def find_word_len4(s):
    return re.findall(r'\b\w{4,}\b', s)

print(find_word_len4("hi hello test a abcde")) 

['hello', 'test', 'abcde']


In [12]:
# 任务11 删除字符串中除字母和数字之外的所有字符
def keep_alnum(s):
    return re.sub(r'[^a-zA-Z0-9]', '', s)

print(keep_alnum("abc!123@xyz_")) 

abc123xyz


In [13]:
# 任务12 删除字符串中的括号及括号内的内容
def remove_bracket_content(s):
    return re.sub(r'\s*\(.*?\)', '', s)

print(remove_bracket_content("example (.com)"))      
print(remove_bracket_content("w3resource"))   
print(remove_bracket_content("github (.com)")) 
print(remove_bracket_content("stackoverflow (.com)"))

example
w3resource
github
stackoverflow


## 从网⻚检索气候数据

访问日期：2026-11-24  

实际使用的 URL：
| 城市 | URL |
|---|---|
| Madison, Wisconsin | https://en.wikipedia.org/wiki/Madison,_Wisconsin |
| Los_Angeles | https://en.wikipedia.org/wiki/Los_Angeles |
| Hyderabad | https://en.wikipedia.org/wiki/Hyderabad |
| Boston | https://en.wikipedia.org/wiki/Boston |

In [14]:
import requests
from bs4 import BeautifulSoup

# 任务1 从维基百科页面中提取气候表格
# 请求头（避免 403 错误）
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

url = "https://en.wikipedia.org/wiki/Madison,_Wisconsin"
response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

# 找到所有的表格并检查
tables = soup.find_all('table')
for i, table in enumerate(tables):
    # 合并空白字符进行检查
    table_text = " ".join(table.get_text().split())
    
    # 定位气候表格
    if "Climate data for" in table_text[:100]:
        print(f"在第 {i+1} 个表格找到了气候数据\n")
        
        # 1. 获取 table 的直接子节点
        # 使用 .children 获取直接子节点，并过滤掉换行符等空文本节点（child.name is not None）
        direct_children = [child for child in table.children if child.name is not None]
        
        # 2. 打印数量和标签名
        print("--- 表格的直接子节点统计 ---")
        print(f"直接子节点数量: {len(direct_children)} 个")
        print("直接子节点的标签名:")
        for child in direct_children:
            print(f" - <{child.name}>")
        
        # 3. 打印完整结构
        print("\n--- 表格的完整 HTML 结构 ---")
        print(table.prettify())
        
        break

在第 2 个表格找到了气候数据

--- 表格的直接子节点统计 ---
直接子节点数量: 1 个
直接子节点的标签名:
 - <tbody>

--- 表格的完整 HTML 结构 ---
<table class="wikitable mw-collapsible" style="width:auto; text-align:center; line-height:1.2em;">
 <tbody>
  <tr>
   <th colspan="14">
    Climate data for Madison, Wisconsin (
    <a href="https://en.wikipedia.org/wiki/Dane_County_Regional_Airport" rel="mw:WikiLink" title="Dane County Regional Airport">
     Dane County Regional Airport
    </a>
    ), 1991–2020 normals,
    <sup about="#mwt217" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-60"}}' id="cite_ref-60" rel="dc:references" typeof="mw:Extension/ref">
     <a data-mw-group="lower-alpha" href="#cite_note-60" id="mwAe8">
      <span class="mw-reflink-text" id="mwAfA">
       <span class="cite-bracket" id="mwAfE">
        [
       </span>
       a
       <span class="cite-bracket" id="mwAfI">
        ]
       </span>
      </span>
     </a>
    </sup>
    extre

In [15]:
# 任务2 提取页面中指定气候数据表格的 Tag 对象
def retrieve_climate_table(url):
    """
    接收一个 URL 字符串。
    若页面中存在气候数据表格，返回对应的 BeautifulSoup Tag 对象；
    若成功读取页面但没有此类表格，返回 None。
    网络错误或 HTTP 错误抛出异常。
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        # 捕获到异常后，打印一下错误信息，然后重新抛出
        print(f"  [网络请求失败] 抛出异常: {type(e).__name__}")
        raise e  
    
    soup = BeautifulSoup(response.text, 'html.parser')
    tables = soup.find_all('table')
    
    # 遍历页面中的所有表格
    for table in tables:
        # 1. 检查 caption 节点
        caption = table.find('caption')
        if caption:
            # 使用 get_text() 并合并连续空白字符
            text = " ".join(caption.get_text().split())
            if "Climate data for" in text:
                # 若有多个，返回最先找到的任意一个
                return table
        
        # 2. 检查表格开头内部文本
        table_text = " ".join(table.get_text().split())
        if "Climate data for" in table_text[:100]: 
            # 若有多个，返回最先找到的任意一个
            return table
            
    # 成功读取页面，但遍历完所有表格都没找到，返回 None
    return None

# 测试
test_urls_task2 = [
    ("Madison, Wisconsin", "https://en.wikipedia.org/wiki/Madison,_Wisconsin"),
    ("Los Angeles", "https://en.wikipedia.org/wiki/Los_Angeles"),
    ("Hyderabad", "https://en.wikipedia.org/wiki/Hyderabad"),
    ("Boston", "https://en.wikipedia.org/wiki/Boston")
]

for city_name, url in test_urls_task2:
    print(f"正在测试: {city_name} ...")
    result = retrieve_climate_table(url)
        
    # 检查返回值类型是否符合要求
    if type(result).__name__ == 'Tag':
        print(f"  成功找到表格！返回类型为: {type(result).__name__}")
        # 打印一点点结果验证
        print(f"     (表格 class: {result.get('class')})")
    elif result is None:
        print("  成功读取页面，但未找到气候表格")
    else:
        print(f"  返回了意外的类型: {type(result)}")


正在测试: Madison, Wisconsin ...
  成功找到表格！返回类型为: Tag
     (表格 class: ['wikitable', 'mw-collapsible'])
正在测试: Los Angeles ...
  成功找到表格！返回类型为: Tag
     (表格 class: ['wikitable', 'mw-collapsible'])
正在测试: Hyderabad ...
  成功找到表格！返回类型为: Tag
     (表格 class: ['wikitable', 'mw-collapsible'])
正在测试: Boston ...
  成功找到表格！返回类型为: Tag
     (表格 class: ['wikitable', 'mw-collapsible'])


In [16]:
# 任务3 提取气候表格的行名
def list_climate_table_row_names(url):
    """
    接收一个城市页面 URL。
    若找到气候数据表格，返回其行名组成的列表；
    若成功读取页面但找不到，返回 None。
    """
    table = retrieve_climate_table(url)
    if not table:
        return None

    row_names = []
    # 常见的月份和表头标识，用于排除列标题行
    months_and_headers = {"jan", "feb", "mar", "apr", "may", "jun", 
                          "jul", "aug", "sep", "oct", "nov", "dec",
                          "month", "year"}
    
    rows = table.find_all('tr')
    for row in rows:
        th_tags = row.find_all('th')
        if not th_tags:
            continue
            
        # 结合 <tr> 与 <th> 的位置判断行名，通常第一个 th 是行名
        first_th = th_tags[0]
        
        # 提取文本：去除首尾空白，并合并连续空白字符
        text = " ".join(first_th.get_text().split())
        
        # 过滤 1：排除空文本或表头月份
        if not text or text.lower() in months_and_headers:
            continue
            
        # 过滤 2：排除 "Climate data for..." 标题行
        if "Climate data for" in text:
            continue
            
        # 过滤 3：排除数据来源行
        if text.lower().startswith("source"):
            continue
            
        # 辅助判断：如果有 scope="row"，则确认是行名；如果没有，根据上面的排除法也可认定为行名
        if first_th.get('scope') == 'row' or first_th.get('scope') is None:
            row_names.append(text)
            
    return row_names

# 测试
test_urls_task3 = [
    ("Madison, Wisconsin", "https://en.wikipedia.org/wiki/Madison,_Wisconsin"),
    ("Los Angeles", "https://en.wikipedia.org/wiki/Los_Angeles"),
    ("Hyderabad", "https://en.wikipedia.org/wiki/Hyderabad"),
    ("Boston", "https://en.wikipedia.org/wiki/Boston")
]

for city_name, url in test_urls_task3:
    print(f"正在测试城市: {city_name}")
    
    # 调用函数获取行名列表
    row_names = list_climate_table_row_names(url)
        
    if row_names is None:
        print("  未找到气候数据表格，返回 None")       
    elif len(row_names) == 0:
        print("  找到了表格，但提取出的行名列表为空")       
    else:
        print(f"  成功提取行名，共 {len(row_names)} 个：")
            
        # 打印前 3 个行名
        for name in row_names[:3]:
            print(f"     - {repr(name)}")

正在测试城市: Madison, Wisconsin
  成功提取行名，共 15 个：
     - 'Record high °F (°C)'
     - 'Mean maximum °F (°C)'
     - 'Mean daily maximum °F (°C)'
正在测试城市: Los Angeles
  成功提取行名，共 12 个：
     - 'Record high °F (°C)'
     - 'Mean maximum °F (°C)'
     - 'Mean daily maximum °F (°C)'
正在测试城市: Hyderabad
  成功提取行名，共 13 个：
     - 'Record high °C (°F)'
     - 'Mean daily maximum °C (°F)'
     - 'Daily mean °C (°F)'
正在测试城市: Boston
  成功提取行名，共 16 个：
     - 'Record high °F (°C)'
     - 'Mean maximum °F (°C)'
     - 'Mean daily maximum °F (°C)'
